# Autoportrait dans le rap — Qwen3-32B

## Auteurs du projet

**Agathe Salmon, Nathan Tessot et Marius Zarka**, étudiants en licence au **CPES Sciences des données, arts et cultures (DAC)** de l'[Université PSL](https://psl.eu/) et du [lycée Louis-le-Grand](https://louislegrand.fr/), pour l'année universitaire **2026–2027**.

Présentation de la formation : [CPES Sciences des données, arts et cultures — PSL / Louis-le-Grand](https://psl.eu/formation/cpes-psl-louis-le-grand).

## Présentation du carnet

Ce carnet a un seul objectif : **annoter les chansons de `Rap/corpus.csv` avec Qwen/Qwen3-32B en mode raisonnement**. Le chemin du fichier, depuis le dossier de ce carnet (`Rap/Autoportrait/`), est `../corpus.csv`. Le corpus réellement étudié n'est pas la base entière mais un **corpus d'étude stratifié par année** : toutes les chansons datées d'avant 2004, puis 500 chansons tirées au hasard pour chaque année de 2004 à 2024 ; les chansons sans année (`NA`), l'année 2025 (incomplète) et la date aberrante 1905 sont écartées. Cela donne **12 938 chansons** (2 438 avant 2004 + 21 × 500). Les petits tests sont tirés **dans ce corpus d'étude**, jamais dans la base entière. Aucune comparaison entre modèles n'est réalisée.

Chaque chanson reçoit une décision parmi `autoportrait`, `pas_autoportrait`, `je_ne_sais_pas`, ainsi qu'une justification. La consigne demande une justification en français, sans citation des paroles. La définition est lue dans [consigne.md](consigne.md), sans modification. Le nettoyage, les messages et le schéma JSON sont réutilisés depuis [annoter.py](annoter.py) afin de garder une consigne unique. Aucun fichier de référence ni ancienne annotation n'est transmis au modèle. Les contrôles techniques vérifient le format et les valeurs, pas le respect sémantique de la consigne.

## Plan du carnet

0. **Configuration** : tous les réglages sont en tête de la première cellule de code, dans le bloc « PARAMÈTRES À RÉGLER », rangés selon les étapes ci-dessous.
1. **Chargement du corpus** : lecture de toute la base `corpus.csv` et nettoyage des paroles (84 821 chansons exploitables).
2. **Constitution du corpus d'étude stratifié** : tirage par année (toutes avant 2004, 500 par année ensuite, NA/1905/2025 écartées) et tableau du nombre de chansons par année, de la base au corpus d'étude.
3. **Échantillonnage : test ou corpus complet** : `taille_echantillon = None` retient tout le corpus d'étude ; un entier tire ce nombre de chansons dans le corpus d'étude pour un test.
4. **Annotation** : client vLLM (4.1), journal, reprise et exports (4.2), puis lancement de l'annotation de la sélection (4.3).

Les étapes 1 à 3 ne lancent **aucune inférence** et ne demandent pas de serveur : on peut vérifier le corpus d'étude et la sélection avant de mobiliser les GPU. Seule la dernière cellule (4.3) envoie des requêtes au modèle.

## Utilisation
1. Choisir un noyau Python avec **pandas** et **tqdm** (par exemple **Rap R05b (PyTorch CPU)**). Les dépendances figurent dans [requirements.txt](requirements.txt) ; si nécessaire, les installer avec le Python du noyau. Exécuter depuis la racine du dépôt, `Rap/` ou `Rap/Autoportrait/`.
2. **Avant de démarrer un serveur**, contrôler `nvidia-smi` : mémoire libre, activité et processus de chaque GPU. Ne pas arrêter les processus d'autres utilisateurs. Choisir un GPU libre avec `CUDA_VISIBLE_DEVICES`. Le noyau du carnet reste un client CPU ; le modèle est chargé par le serveur vLLM, pas par ce carnet.
3. Disposer de **deux instances indépendantes de vLLM / Qwen/Qwen3-32B**, avec `--reasoning-parser qwen3` : GPU physique 1 sur `http://127.0.0.1:8016/v1` et GPU physique 2 sur `http://127.0.0.1:8017/v1`. Chaque GPU porte un modèle complet ; il ne s'agit pas de tensor parallelism. Le carnet ne démarre ni n'arrête les serveurs et ne télécharge pas de modèle. `QWEN_BASE_URLS` accepte une liste d'URL séparées par des virgules ; à défaut, `QWEN_BASE_URL` permet de choisir une seule adresse. `QWEN_API_KEY` fournit une clé commune aux serveurs.
4. Régler le bloc « PARAMÈTRES À RÉGLER » : les strates (`annee_exhaustive_avant = 2004`, `chansons_par_annee = 500`, `annees_exclues = {1905, 2025}`), puis `taille_echantillon`, qui vaut **`None` pour tout le corpus d'étude** ou un entier pour un test (**220 actuellement**). Une taille invalide ou supérieure au corpus d'étude est refusée ; elle n'est jamais remplacée silencieusement par une autre sélection.
5. **Exécuter toutes les cellules dans l'ordre depuis la configuration**, afin de recréer la file de créneaux et de remplacer les anciennes fonctions dans le noyau. Vérifier le tableau par année (étape 2) et le périmètre annoncé (étape 3) avant la dernière cellule, qui annote directement la sélection. Avec `None`, elle lance les 12 938 chansons du corpus d'étude (environ 3 h 30 à 1 chanson/s). Consulter ensuite les décisions, justifications et raisonnements dans les exports. Une relance reprend les réussites compatibles et retente les erreurs. Ne pas modifier la configuration pendant qu'une annotation est en cours.

Les résultats vont dans `data/results/`, les métriques dans `data/performance/`, sous ce dossier. Les exports se nomment `autoportrait_qwen_annotation_{n}.csv` et `.json`, où `n` est le nombre de chansons sélectionnées (220 actuellement, 12 938 avec `None`). Un nouveau lot de même taille remplace ces exports ; les journaux JSONL restent distincts par empreinte. Ils sont sauvegardés après chaque réponse ; une relance reprend les réussites et retente les erreurs. **Une erreur réseau, HTTP ou JSON n'est jamais une décision `je_ne_sais_pas`.**

## Paramètres et limites

Ne pas confondre les tailles : **`taille_echantillon=220` choisit le nombre de chansons du test** (ou `None` pour tout le corpus d'étude) ; **`chansons_par_annee=500` fixe la taille des strates** du corpus d'étude ; **`workers_par_gpu=12` fixe le nombre de requêtes envoyées en parallèle à chaque serveur, donc à chaque GPU** ; **`taille_lot=128` limite les tâches soumises à la fois**. Le total des workers est calculé automatiquement : **24 workers pour deux GPU** (12 × 2). Un worker n'est réellement actif que si le serveur vLLM l'accepte : au-delà de son option `--max-num-seqs`, les requêtes supplémentaires attendent dans la file du serveur sans accélérer le calcul. Il faut donc garder `workers_par_gpu` inférieur ou égal à `--max-num-seqs` ; avec Qwen3-32B sur un GPU de 80 Go et un contexte de 16 384 jetons, 12 à 16 sont envisageables. **Les deux serveurs actuels sont lancés avec `--max-num-seqs 12`** : les 24 workers sont donc tous actifs ; passer à 16 par GPU impose de relancer les serveurs avec `--max-num-seqs 16`. Une file de créneaux partagée attribue chaque chanson à un serveur disposant d'une place ; le créneau est rendu même en cas d'erreur. Un serveur rapide peut ainsi traiter davantage de chansons, sans partage rigide en deux moitiés. Une tentative supplémentaire reste sur le même serveur. Un seul client centralise le journal et les exports, dans l'ordre du corpus. La colonne `serveur` permet d'identifier l'instance utilisée ; les métriques enregistrent les adresses et la concurrence. Aucun basculement silencieux n'est effectué si un serveur manque : tous sont vérifiés avant une nouvelle inférence.

Le raisonnement Qwen3 est activé (`enable_thinking=True`), avec une température de 0 et une graine de 42. Ces réglages facilitent la reproductibilité, sans garantir une identité absolue entre versions de serveur ou matériels. Le budget de **8192 tokens** couvre **le raisonnement et la réponse finale réunis** ; une génération coupée n'est jamais acceptée. Le délai de 600 secondes s'applique à chaque requête, avec deux tentatives au maximum ; ce n'est pas une limite de durée pour toute la sélection.

Le contexte serveur doit accueillir les messages d'entrée et le budget de sortie. Un contexte de 16384 tokens ne garantit pas que toute chanson longue y tiendra. Les paroles sont envoyées intégralement après nettoyage : aucun filtre supplémentaire, aucune troncature silencieuse. Un dépassement de contexte est enregistré comme erreur à corriger, pas comme une hésitation du modèle. La mémoire GPU et l'espace disque disponible restent à surveiller lors d'un traitement complet.

In [1]:
# ====================== PARAMÈTRES À RÉGLER ======================
# --- 2. Corpus d'étude stratifié par année (colonne year du corpus) ---
annee_exhaustive_avant = 2004   # toutes les chansons des années antérieures
chansons_par_annee = 500        # tirage par année à partir de ce seuil
annees_exclues = {1905, 2025}   # 1905 : date aberrante ; 2025 : année incomplète ; NA toujours écartées
# --- 3. Échantillonnage dans le corpus d'étude ---
# None = corpus d'étude complet (≈ 12 900 chansons) ; un entier (ex. 10, 220) = test tiré dans le corpus d'étude.
taille_echantillon = None
# --- 4. Annotation ---
# Nombre de workers PAR GPU (= requêtes envoyées en parallèle à chaque serveur vLLM).
# Total des workers = workers_par_gpu × nombre de serveurs (2 GPU → 2 × 12 = 24).
# Au-delà de --max-num-seqs du serveur vLLM, les requêtes en trop attendent en file :
# garder workers_par_gpu ≤ --max-num-seqs (12 sur les serveurs actuels ; 16 impose de les relancer).
workers_par_gpu = 12
# Nombre maximal de chansons soumises à la fois (mémoire) ; ne change pas le total.
taille_lot = 128
# ==================================================================

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display
from queue import Queue
import fcntl
import hashlib
import importlib.util
import json
import os
import time
import urllib.error
import urllib.request

import pandas as pd
from tqdm.auto import tqdm

# Accepte un lancement depuis la racine du dépôt, Rap ou Autoportrait.
candidats = [Path.cwd(), Path.cwd() / 'Autoportrait', Path.cwd() / 'Rap' / 'Autoportrait']
dossiers = [p.resolve() for p in candidats if (p / 'consigne.md').is_file() and (p / 'annoter.py').is_file()]
if len(dossiers) != 1:
    raise RuntimeError('Lancer le carnet depuis la racine du dépôt, Rap ou Rap/Autoportrait.')
dossier = dossiers[0]
corpus_path = dossier.parent / 'corpus.csv'
consigne_path = dossier / 'consigne.md'
resultats_dir = dossier / 'data' / 'results'
performance_dir = dossier / 'data' / 'performance'
cumul_path = performance_dir / 'perfGPU_autoportrait.csv'

urls_configurees = os.environ.get('QWEN_BASE_URLS', os.environ.get(
    'QWEN_BASE_URL', 'http://127.0.0.1:8016/v1,http://127.0.0.1:8017/v1'))
base_urls = [url.strip().rstrip('/') for url in urls_configurees.split(',')]
if any(not url.startswith(('http://', 'https://')) for url in base_urls) or len(set(base_urls)) != len(base_urls):
    raise ValueError('Configurer des URL HTTP(S) distinctes dans QWEN_BASE_URLS.')
modele = 'Qwen/Qwen3-32B'
cle_api = os.environ.get('QWEN_API_KEY', 'EMPTY')
graine = 42
concurrence_par_serveur = workers_par_gpu
concurrence = concurrence_par_serveur * len(base_urls)
serveurs_disponibles = Queue()
for _ in range(concurrence_par_serveur):
    for url in base_urls:
        serveurs_disponibles.put(url)
timeout_secondes = 600
max_tokens = 8192
enable_thinking = True
max_essais = 2

if taille_echantillon is not None and (type(taille_echantillon) is not int or taille_echantillon <= 0):
    raise ValueError("taille_echantillon doit être un entier positif, ou None pour tout le corpus d'étude.")
if type(annee_exhaustive_avant) is not int or type(chansons_par_annee) is not int or chansons_par_annee <= 0:
    raise ValueError('annee_exhaustive_avant et chansons_par_annee doivent être des entiers, le second positif.')
if any(type(a) is not int for a in annees_exclues):
    raise ValueError('annees_exclues doit contenir des années entières.')
if type(workers_par_gpu) is not int:
    raise ValueError('workers_par_gpu doit être un entier positif.')
if min(taille_lot, concurrence, timeout_secondes, max_tokens, max_essais) <= 0:
    raise ValueError('Les tailles, délais et nombres de tentatives doivent être positifs.')
spec = importlib.util.spec_from_file_location('autoportrait_source', dossier / 'annoter.py')
if spec is None or spec.loader is None:
    raise ImportError('Impossible de charger annoter.py.')
source = importlib.util.module_from_spec(spec)
spec.loader.exec_module(source)
consigne = consigne_path.read_text(encoding='utf-8').strip()
if not consigne:
    raise ValueError('La consigne est vide.')
print('Dossier :', dossier)
print('Modèle :', modele, '| endpoints :', base_urls, '| raisonnement :', enable_thinking)
print('Workers :', concurrence, 'au total |', workers_par_gpu, 'par GPU/serveur')

Dossier : /home/cbenavent/test/Rap/Autoportrait
Modèle : Qwen/Qwen3-32B | endpoints : ['http://127.0.0.1:8016/v1', 'http://127.0.0.1:8017/v1'] | raisonnement : True
Workers : 24 au total | 12 par GPU/serveur


## 1. Chargement du corpus

### Fichier d'entrée et identifiants

La seule source de chansons est **[Rap/corpus.csv](../corpus.csv)**, lu directement avec la fonction commune de [annoter.py](annoter.py). Le code construit son chemin depuis le dossier du carnet, et non depuis un chemin relatif dépendant du répertoire d'exécution. Les colonnes attendues sont `artist`, `title`, `year` et `lyrics`. `source_row_index` conserve la **position de l'enregistrement de données, à partir de zéro**, dans ce fichier. Ce n'est ni un numéro de ligne physique (les paroles peuvent contenir des retours à la ligne), ni un `document_id`. Cet identifiant permet de rattacher chaque annotation à l'enregistrement original ; un corpus réordonné peut changer les identifiants et l'empreinte de reprise.

« Corpus complet » désigne ici **toutes les lignes avec des paroles exploitables**, sans restriction sur le titre, l'artiste, l'année ou le nombre de mots. Comme dans le script commun, les paroles absentes, vides ou égales à `NA` ne sont pas retenues. Le nettoyage retire les résidus déjà gérés par ce script, sans tronquer les textes. Les chansons devenues vides après nettoyage sont également exclues et leur nombre est affiché explicitement. Si aucun texte exploitable ne reste, l'exécution s'arrête.

### Ce qui est conservé ou envoyé

Les chansons restent dans l'ordre du corpus. Les métadonnées exportées sont la position source, l'artiste, le titre, l'année et le nombre de mots après nettoyage. Aucune ancienne annotation n'est utilisée pour préparer les textes ou les messages ; seuls les journaux compatibles pourront ensuite être relus pour la reprise. Le modèle reçoit uniquement la consigne et les paroles nettoyées. L'aperçu ci-dessous n'affiche pas les paroles. **Cette étape lit et prépare toute la base exploitable en mémoire**, même pour un petit test : le corpus d'étude est constitué ensuite à l'étape 2, puis l'échantillon de test ou complet à l'étape 3. La barre de préparation commence après la lecture du CSV ; elle ne mesure donc pas cette lecture initiale. **Aucune requête d'annotation n'est envoyée à cette étape.**

In [2]:
def charger_chansons():
    corpus = source.lire_corpus(corpus_path, None)
    chansons = []
    vides = 0
    for position, chanson in tqdm(corpus.items(), total=len(corpus), desc='Préparation du corpus', unit='chansons'):
        paroles = source.nettoyer(chanson['paroles'])
        if not paroles:
            vides += 1
            continue
        chansons.append({**chanson, 'source_row_index': position, 'paroles': paroles,
                         'mots_nettoyes': len(paroles.split())})
    print(f'{len(chansons)} chansons prêtes ; {vides} textes supplémentaires vides après nettoyage.')
    if not chansons:
        raise ValueError('Aucune chanson avec des paroles exploitables dans le corpus.')
    return chansons

chansons = charger_chansons()
display(pd.DataFrame(chansons).drop(columns='paroles').head())
print('Aucune inférence lancée ; aucun texte tronqué.')


Préparation du corpus:   0%|          | 0/84852 [00:00<?, ?chansons/s]

84821 chansons prêtes ; 31 textes supplémentaires vides après nettoyage.


,artiste,titre,annee,source_row_index,mots_nettoyes
0,Dry,94310,2012,0,673
1,Mafia K’1 Fry,Au bon vieux temps,2007,1,837
2,DJ Hamida,Attrape-Moi Si Tu Peux,,2,306
3,Kery James,94 c’est le Barça Remix,,3,481
4,Dry,14 ans déjà,2013,4,576


Aucune inférence lancée ; aucun texte tronqué.


## 2. Constitution du corpus d'étude stratifié

### Règle de stratification

Le corpus d'étude est constitué une fois pour toutes à partir des chansons exploitables de l'étape 1 (paroles non vides après nettoyage), selon la colonne `year` de `corpus.csv` :

- **avant 2004** (`annee_exhaustive_avant`) : **toutes** les chansons sont gardées, car ces années sont peu représentées (2 438 chansons de 1990 à 2003) ;
- **de 2004 à 2024** : **500 chansons par année** (`chansons_par_annee`), tirées sans remise. Chaque année compte au moins 524 chansons exploitables ; si une année en comptait moins de 500, toutes seraient gardées et la cellule le signalerait ;
- **écartées** : les chansons sans année (`NA`, 9 594 chansons exploitables), l'année 2025 (une seule chanson, année incomplète) et 1905 (une chanson, date manifestement erronée), via `annees_exclues`.

Le total est de **12 938 chansons sur 84 821** exploitables, réparties sur 35 années (1990–2024). Le tirage de chaque année utilise la graine `42 + année` : il est reproductible et indépendant des autres années, si bien que modifier une strate ne change pas le tirage des autres. Les chansons retenues sont remises dans l'ordre du corpus. L'année n'est ni corrigée ni déduite par le modèle ; une valeur non numérique est traitée comme `NA`.

Ce plan **sur-représente volontairement les années anciennes** : une proportion calculée sur l'ensemble du corpus d'étude n'est donc pas celle de la base. Pour estimer une fréquence globale, il faut pondérer chaque année par son poids réel (colonne `chansons_exploitables` du tableau ci-dessous) ; les comparaisons **année par année** restent directement lisibles.

### Tableau du nombre de chansons par année

La cellule affiche et enregistre dans `data/results/corpus_chansons_par_annee.csv` un tableau qui suit chaque année de la base jusqu'au corpus d'étude :

- `chansons_base` : toutes les lignes de `corpus.csv` pour cette année, y compris celles sans paroles ;
- `chansons_exploitables` : les chansons dont les paroles restent non vides après nettoyage (étape 1) ;
- `corpus_etude` : les chansons retenues dans le corpus d'étude (toutes avant 2004, 500 ensuite, 0 pour `NA`, 1905 et 2025) ;
- `part_base` : la part de l'année dans toute la base, en pourcentage.

Les années sont triées dans l'ordre chronologique, la ligne **`NA`** (année absente ou vide) vient ensuite, puis une ligne `Total` qui doit égaler le nombre de lignes de la base (85 023), le nombre de chansons exploitables (84 821) et la taille du corpus d'étude (12 938) ; sinon la cellule s'arrête. Le tableau ne mesure pas la fréquence des autoportraits. **Aucune inférence n'est lancée à cette étape.**

In [3]:
def annee_entiere(chanson):
    texte = str(chanson.get('annee') or '').strip()
    return int(texte) if texte.isdigit() else None


def constituer_corpus_etude(chansons):
    """Toutes les chansons avant le seuil, chansons_par_annee ensuite ; NA et années exclues écartées."""
    par_annee = {}
    for position, chanson in enumerate(chansons):
        annee = annee_entiere(chanson)
        if annee is not None and annee not in annees_exclues:
            par_annee.setdefault(annee, []).append(position)
    retenues = []
    for annee, positions in sorted(par_annee.items()):
        if annee < annee_exhaustive_avant or len(positions) <= chansons_par_annee:
            retenues += positions
        else:
            retenues += pd.Series(positions).sample(n=chansons_par_annee, random_state=graine + annee).tolist()
    return [chansons[i] for i in sorted(retenues)]


def compter_par_annee(annees):
    annees = pd.to_numeric(pd.Series(annees, dtype='string').replace({'NA': pd.NA, '': pd.NA}), errors='coerce')
    return annees.astype('Int64').astype('string').fillna('NA').value_counts()


corpus_etude = constituer_corpus_etude(chansons)
base = pd.read_csv(corpus_path, usecols=['year'], dtype='string', keep_default_na=False)
tableau_annees = pd.DataFrame({
    'chansons_base': compter_par_annee(base['year']),
    'chansons_exploitables': compter_par_annee([c['annee'] for c in chansons]),
    'corpus_etude': compter_par_annee([c['annee'] for c in corpus_etude]),
}).fillna(0).astype(int)
ordre = sorted(a for a in tableau_annees.index if a != 'NA') + (['NA'] if 'NA' in tableau_annees.index else [])
tableau_annees = tableau_annees.reindex(ordre)
tableau_annees.loc['Total'] = tableau_annees.sum()
tableau_annees['part_base'] = (100 * tableau_annees['chansons_base'] / len(base)).round(2)
tableau_annees.index.name = 'annee'
totaux = tableau_annees.loc['Total']
if (totaux['chansons_base'], totaux['chansons_exploitables'], totaux['corpus_etude']) != (len(base), len(chansons), len(corpus_etude)):
    raise RuntimeError('Le tableau par année ne couvre pas toute la base.')
annees_path = resultats_dir / 'corpus_chansons_par_annee.csv'
annees_path.parent.mkdir(parents=True, exist_ok=True)
tableau_annees.to_csv(annees_path, encoding='utf-8')

etude = tableau_annees.loc[[a for a in ordre if a != 'NA'], 'corpus_etude']
etude = etude[etude > 0]
annees_etude = etude.index.astype(int)
incompletes = [a for a, n in zip(annees_etude, etude) if a >= annee_exhaustive_avant and n < chansons_par_annee]
print(f"Corpus d'étude : {len(corpus_etude)} chansons sur {len(chansons)} exploitables, "
      f"{len(etude)} années ({annees_etude.min()}–{annees_etude.max()}).")
print(f'  avant {annee_exhaustive_avant} : {int(etude[annees_etude < annee_exhaustive_avant].sum())} chansons (toutes) ; '
      f'à partir de {annee_exhaustive_avant} : {chansons_par_annee} par année ; '
      f'écartées : NA et {sorted(annees_exclues)}.')
if incompletes:
    print('Années avec moins de', chansons_par_annee, 'chansons (toutes gardées) :', incompletes)
print('Tableau par année enregistré :', annees_path)
with pd.option_context('display.max_rows', None):
    display(tableau_annees)

Corpus d'étude : 12938 chansons sur 84821 exploitables, 35 années (1990–2024).
  avant 2004 : 2438 chansons (toutes) ; à partir de 2004 : 500 par année ; écartées : NA et [1905, 2025].
Tableau par année enregistré : /home/cbenavent/test/Rap/Autoportrait/data/results/corpus_chansons_par_annee.csv


,chansons_base,chansons_exploitables,corpus_etude,part_base
annee,,,,
1905,1,1,0,0.00
1990,21,21,21,0.02
1991,34,34,34,0.04
1992,10,10,10,0.01
1993,51,50,50,0.06
1994,37,37,37,0.04
1995,139,138,138,0.16
1996,112,110,110,0.13
1997,212,212,212,0.25


## 3. Échantillonnage : test ou corpus complet

Cette étape choisit, **dans le corpus d'étude**, les chansons qui seront annotées à l'étape 4 :

- avec `taille_echantillon = None`, **les 12 938 chansons du corpus d'étude** sont retenues ; le périmètre des journaux est `strates` ;
- avec un entier (10, 220…), la cellule tire ce nombre de chansons au hasard dans le corpus d'étude (graine 42), pour un test rapide ; le périmètre est `strates_test`. Ce petit échantillon n'est pas lui-même stratifié : il reflète à peu près la composition du corpus d'étude, donc avec une part d'années anciennes plus forte que dans la base.

Une taille non entière, nulle ou supérieure au corpus d'étude est refusée. La même graine et la même taille donnent le même test ; un test de 10 et un test de 220 sont deux tirages différents. Les chansons restent dans l'ordre du corpus.

La cellule affiche le périmètre, le nombre de chansons, les années représentées et le nom de l'export prévu (`autoportrait_qwen_annotation_{n}`). **Elle ne lance aucune inférence** : vérifier ce périmètre, surtout avant un lancement complet (environ 3 h 30), puis passer à l'étape 4. Changer `taille_echantillon` impose de réexécuter la configuration puis cette cellule ; les étapes 1 et 2 n'ont pas besoin d'être relancées.

Comme la sélection fait partie de l'empreinte de reprise, les anciens journaux tirés dans toute la base (périmètres `echantillon` ou `corpus`) ne sont pas réutilisés.

In [4]:
if taille_echantillon is None:
    selection = corpus_etude
    perimetre = 'strates'
else:
    if type(taille_echantillon) is not int or not 0 < taille_echantillon <= len(corpus_etude):
        raise ValueError(f"taille_echantillon doit être un entier entre 1 et {len(corpus_etude)} (corpus d'étude), ou None.")
    indices = sorted(pd.Series(range(len(corpus_etude))).sample(n=taille_echantillon, random_state=graine).tolist())
    selection = [corpus_etude[i] for i in indices]
    perimetre = 'strates_test'
annees_selection = pd.Series([annee_entiere(c) for c in selection])
mode_lancement = "corpus d'étude complet" if taille_echantillon is None else 'test'
print(f"Périmètre : {perimetre} ({mode_lancement}) ; {len(selection)} chansons sur {len(corpus_etude)} du corpus d'étude.")
print(f'  {annees_selection.nunique()} années représentées ({annees_selection.min()}–{annees_selection.max()}) ; '
      f'{int((annees_selection < annee_exhaustive_avant).sum())} chansons avant {annee_exhaustive_avant}.')
print(f'Export prévu : autoportrait_qwen_annotation_{len(selection)} — aucune inférence lancée à cette étape.')

Périmètre : strates (corpus d'étude complet) ; 12938 chansons sur 12938 du corpus d'étude.
  35 années représentées (1990–2024) ; 2438 chansons avant 2004.
Export prévu : autoportrait_qwen_annotation_12938 — aucune inférence lancée à cette étape.


## 4. Annotation

Cette étape regroupe tout ce qui touche au modèle : le client vLLM et ses contrôles (4.1), le journal de reprise et les exports (4.2), puis le lancement de l'annotation de la sélection de l'étape 3 (4.3). Les cellules 4.1 et 4.2 définissent seulement des fonctions ; **seule la cellule 4.3 envoie des requêtes**.

### 4.1 Client vLLM et validation stricte

#### Contrat serveur et réponse finale

Avant toute nouvelle inférence, **chaque serveur configuré** doit annoncer exactement `Qwen/Qwen3-32B` sur son endpoint `/v1/models`. Si une instance est inaccessible ou sert un autre modèle, le traitement s'arrête avant l'envoi des chansons ; il ne continue pas silencieusement sur un seul GPU. Les deux instances doivent utiliser les mêmes poids, le même parseur de raisonnement et les mêmes réglages de génération ; le contrôle du nom du modèle ne suffit pas à certifier cette identité. Chaque requête active le raisonnement dans le gabarit de conversation et impose le schéma JSON de [annoter.py](annoter.py) à la réponse finale.

Une réponse réussie doit avoir une génération terminée (`finish_reason='stop'`), un objet JSON avec **exactement** `etiquette` et `justification`, une étiquette autorisée et une justification textuelle non vide. En mode raisonnement, elle doit également comporter un champ séparé `reasoning` ou `reasoning_content` non vide. Le parseur serveur `--reasoning-parser qwen3` assure cette séparation : le raisonnement n'est pas extrait approximativement du JSON final. Il est conservé dans la colonne `raisonnement`, distincte de la justification courte.

#### Erreurs et tentatives

Les erreurs réseau, réponses mal formées et erreurs HTTP transitoires (408, 429, 500, 502, 503, 504) sont retentées dans la limite de `max_essais`, avec une attente entre les tentatives. Une erreur HTTP permanente, notamment un contexte trop court, n'est pas retentée immédiatement. Après échec, la chanson conserve `statut='erreur'`, un message explicite et une étiquette vide. **`je_ne_sais_pas` reste une décision du modèle, jamais un substitut à une panne.** Une réponse syntaxiquement valide ne garantit pas la qualité de la décision : celle-ci relève de la revue humaine.

In [5]:
ouvreur = urllib.request.build_opener(urllib.request.ProxyHandler({}))

def appeler_api(route, corps=None, *, serveur):
    requete = urllib.request.Request(
        serveur + route,
        data=None if corps is None else json.dumps(corps).encode('utf-8'),
        headers={'Content-Type': 'application/json', 'Authorization': f'Bearer {cle_api}'},
    )
    with ouvreur.open(requete, timeout=timeout_secondes) as reponse:
        return json.loads(reponse.read().decode('utf-8'))

def verifier_endpoint(serveur):
    try:
        reponse = appeler_api('/models', serveur=serveur)
    except urllib.error.HTTPError as exc:
        raise RuntimeError(f'Serveur vLLM : HTTP {exc.code} sur {serveur}/models. Vérifier la configuration et la clé API.') from exc
    except (urllib.error.URLError, OSError) as exc:
        raise RuntimeError(
            f'Serveur Qwen injoignable sur {serveur}. Démarrer vLLM avec --reasoning-parser qwen3 '
            'sur un GPU libre, attendre sa disponibilité, puis relancer la cellule. '
            'Le notebook ne démarre pas le serveur automatiquement.') from exc
    modeles = [item['id'] for item in reponse['data']]
    if modele not in modeles:
        raise RuntimeError(f'{modele} non servi sur {serveur}. Modèles disponibles : {modeles}')
    print('Serveur disponible :', serveur, '|', modele)

def verifier_serveur():
    for serveur in base_urls:
        verifier_endpoint(serveur)

def valider_annotation(objet, raisonnement):
    if enable_thinking and (not isinstance(raisonnement, str) or not raisonnement.strip()):
        raise ValueError('Raisonnement absent : vérifier enable_thinking et --reasoning-parser qwen3 côté serveur.')
    if not isinstance(objet, dict) or set(objet) != {'justification', 'etiquette'}:
        raise ValueError('Deux clés exactes attendues : justification et etiquette.')
    if objet['etiquette'] not in source.ETIQUETTES:
        raise ValueError(f"Étiquette inconnue : {objet['etiquette']!r}")
    if not isinstance(objet['justification'], str) or not objet['justification'].strip():
        raise ValueError('Justification vide ou non textuelle.')
    return {'justification': objet['justification'].strip(), 'etiquette': objet['etiquette'],
            'raisonnement': raisonnement.strip() if isinstance(raisonnement, str) else ''}

def valider_reponse(reponse):
    choix = reponse['choices'][0]
    if choix.get('finish_reason') != 'stop':
        raise ValueError(f"Génération incomplète : {choix.get('finish_reason')!r}")
    message = choix['message']
    if not isinstance(message['content'], str):
        raise ValueError('Contenu absent ou non textuel.')
    return valider_annotation(json.loads(message['content']),
                              message.get('reasoning') or message.get('reasoning_content'))

def annoter_chanson(chanson):
    serveur = serveurs_disponibles.get()
    try:
        return {**annoter_sur_serveur(chanson, serveur), 'serveur': serveur}
    finally:
        serveurs_disponibles.put(serveur)

def annoter_sur_serveur(chanson, serveur):
    debut = time.perf_counter()
    corps = {
        'model': modele,
        'messages': source.messages(consigne, chanson['paroles'], False),
        'temperature': 0, 'seed': graine, 'max_tokens': max_tokens,
        'chat_template_kwargs': {'enable_thinking': enable_thinking},
        **source.CONTRAINTES['json_schema'],
    }
    for essai in range(1, max_essais + 1):
        try:
            reponse = appeler_api('/chat/completions', corps, serveur=serveur)
            annotation = valider_reponse(reponse)
            return {**annotation, 'statut': 'ok', 'erreur': '', 'essais': essai,
                    'secondes': time.perf_counter() - debut}
        except urllib.error.HTTPError as exc:
            erreur = f'HTTP {exc.code} : {exc.read().decode("utf-8", "replace")[:1000]}'
            if exc.code not in {408, 429, 500, 502, 503, 504}:
                break
        except (OSError, ValueError, KeyError, IndexError, TypeError) as exc:
            erreur = f'{type(exc).__name__} : {exc}'
        if essai < max_essais:
            time.sleep(2 * essai)
    return {'justification': '', 'etiquette': '', 'raisonnement': '', 'statut': 'erreur', 'erreur': erreur,
            'essais': essai, 'secondes': time.perf_counter() - debut}


### 4.2 Journal, reprise, progression et exports

#### Reprendre sans mélanger les expériences

Une empreinte SHA-256 couvre le modèle, la liste des URL, la consigne, le script commun, les paramètres de génération et les chansons sélectionnées. La version multi-serveurs utilise de nouveaux journaux : les anciens résultats ne sont pas effacés ni repris comme un benchmark à deux GPU. La concurrence, la taille des lots techniques et les délais ne font pas partie de cette empreinte : les ajuster ne force pas à réannoter les réussites. Les noms techniques des journaux conservent le mode, le périmètre, la taille et l'empreinte ; ils ne sont pas les noms simplifiés des exports destinés à la lecture. Cette empreinte ne certifie pas la version du serveur ni ses poids effectifs.

Chaque réponse, y compris une erreur, est écrite et synchronisée sur disque dès sa réception. À la relance, les réussites compatibles sont vérifiées puis reprises ; seules les chansons absentes ou en erreur sont soumises au serveur. Un journal endommagé ou incompatible provoque une erreur explicite et n'est pas ignoré. Si tout est déjà réussi, les exports peuvent être régénérés **sans serveur et sans GPU**. Un verrou sur le nom de sortie empêche deux lots de même taille d'écrire simultanément les mêmes exports.

#### Sorties et ordre des lignes

À la fin du traitement, les fichiers lisibles sont `data/results/autoportrait_qwen_annotation_{n}.csv` et `.json`, sous `Rap/Autoportrait/`, avec une ligne par chanson sélectionnée, **erreurs comprises**, dans l'ordre du corpus. **`n` est la taille totale de la sélection**, pas le nombre de réussites ni `taille_lot`. Le fichier `_erreurs.csv` isole les échecs et reste présent même s'il ne contient que les en-têtes. Une autre sélection de même taille remplace ces exports lisibles ; les journaux distincts permettent de retrouver les résultats de chaque expérience. Les paroles ne sont pas recopiées dans les exports, mais le raisonnement peut en reprendre des éléments : ces fichiers restent des données locales à protéger.

#### Progression et performances

La barre affiche l'avancement, le nombre d'erreurs, le débit en chansons/s et une ETA estimée à partir de la progression observée ; cette ETA n'est pas une durée garantie. Le temps et le débit sont **mesurés côté client**, pas sur le GPU. Les métriques CSV, rangées dans `data/performance/`, distinguent les réussites reprises des chansons traitées pendant la session. Une reprise complète n'est donc pas un nouveau benchmark d'inférence. Une relance remplace la synthèse du même journal par celle de la nouvelle session, mais **ajoute une ligne au fichier unique `perfGPU_autoportrait.csv`**, quelle que soit la taille de l'échantillon, sans effacer les essais précédents.

L'historique possède exactement les colonnes **`date,modele,batch,concurrent,n,chansons/s`**. `date` est l'horodatage UTC d'enregistrement ; `modele` identifie le modèle ; `batch` correspond à `taille_lot` ; `concurrent` est le nombre total de workers, tous serveurs réunis ; `n` est l'effectif total sélectionné, erreurs comprises. `chansons/s` est le nombre de chansons nouvellement traitées, erreurs comprises, divisé par la durée client de la session. Les chansons reprises du cache n'entrent pas dans ce débit ; si toute la sélection vient du cache, le champ reste vide pour éviter une fausse mesure d'inférence. Les synthèses techniques conservent les détails des erreurs et des reprises. Les mesures historiques de 100 et 300 chansons sont regroupées dans ce fichier ; la date inconnue du premier essai reste vide. Malgré le nom `perfGPU`, aucune durée GPU n'est mesurée. Les distributions d'étiquettes excluent les erreurs techniques ; aucune précision, mémoire GPU ou émission carbone n'est déduite de ces mesures.

En cas d'interruption, les réponses déjà journalisées restent disponibles. Les tâches en attente sont annulées lorsque possible ; les requêtes déjà parties peuvent finir avant que la cellule rende la main. **Les exports CSV/JSON ne sont pas actualisés après chaque réponse** : avant la fin du traitement, ils peuvent être absents ou correspondre à une exécution antérieure. Le journal JSONL est la source de reprise pendant le calcul. Relancer le même périmètre pour poursuivre et régénérer les exports finaux.

In [6]:
def empreinte(selection):
    configuration = {
        'version_pipeline': 3, 'modele': modele, 'urls': base_urls, 'consigne': consigne,
        'annoter_sha256': hashlib.sha256((dossier / 'annoter.py').read_bytes()).hexdigest(),
        'temperature': 0, 'seed': graine, 'max_tokens': max_tokens,
        'enable_thinking': enable_thinking, 'contrainte': source.CONTRAINTES['json_schema'],
        'selection': selection,
    }
    return hashlib.sha256(json.dumps(configuration, ensure_ascii=False, sort_keys=True).encode()).hexdigest()

def executer_annotation(selection, perimetre):
    if not selection:
        raise ValueError('Sélection vide.')
    resultats_dir.mkdir(parents=True, exist_ok=True)
    performance_dir.mkdir(parents=True, exist_ok=True)
    signature = empreinte(selection)
    mode = 'reason' if enable_thinking else 'sans_reason'
    prefixe = f'autoportrait_qwen32_{mode}_{perimetre}_n{len(selection)}_{signature[:12]}'
    prefixe_sortie = f'autoportrait_qwen_annotation_{len(selection)}'
    journal = resultats_dir / f'{prefixe}.jsonl'
    ids = [c['source_row_index'] for c in selection]
    ids_selection = set(ids)
    if len(ids) != len(ids_selection):
        raise ValueError('Sélection avec doublons.')
    debut = time.perf_counter()
    with (resultats_dir / f'{prefixe_sortie}.lock').open('a') as verrou:
        try:
            fcntl.flock(verrou, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as exc:
            raise RuntimeError('Un lot de même taille écrit déjà ces fichiers de sortie.') from exc
        resultats = {}
        if journal.exists():
            with journal.open(encoding='utf-8') as fichier:
                for numero, ligne in enumerate(fichier, 1):
                    try:
                        resultat = json.loads(ligne)
                    except json.JSONDecodeError as exc:
                        raise ValueError(f'Journal endommagé : {journal}, ligne {numero}. Réparer avant reprise.') from exc
                    if resultat['empreinte'] != signature or resultat['source_row_index'] not in ids_selection:
                        raise ValueError('Journal incompatible avec la sélection courante.')
                    if resultat['statut'] not in {'ok', 'erreur'}:
                        raise ValueError('Statut invalide dans le journal.')
                    if resultat['statut'] == 'ok':
                        valider_annotation({k: resultat[k] for k in ('justification', 'etiquette')},
                                           resultat.get('raisonnement'))
                    resultats[resultat['source_row_index']] = resultat
        restantes = [c for c in selection if resultats.get(c['source_row_index'], {}).get('statut') != 'ok']
        reprises = len(selection) - len(restantes)
        print(f'{perimetre} : {reprises} réussites reprises ; {len(restantes)} chansons à traiter.')
        if restantes:
            verifier_serveur()
        traitees = erreurs = 0
        with journal.open('a', encoding='utf-8') as fichier:
            with ThreadPoolExecutor(max_workers=concurrence) as executeur:
                taches = {}
                try:
                    with tqdm(total=len(selection), initial=reprises, desc=f'Annotation {perimetre}',
                              unit='chansons', mininterval=0.5) as barre:
                        for debut_lot in range(0, len(restantes), taille_lot):
                            lot = restantes[debut_lot:debut_lot + taille_lot]
                            taches = {executeur.submit(annoter_chanson, c): c for c in lot}
                            for tache in as_completed(taches):
                                chanson = taches[tache]
                                resultat = {**{k: v for k, v in chanson.items() if k != 'paroles'},
                                            **tache.result(), 'modele': modele, 'empreinte': signature,
                                            'date_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
                                fichier.write(json.dumps(resultat, ensure_ascii=False) + '\n')
                                fichier.flush()
                                os.fsync(fichier.fileno())
                                resultats[chanson['source_row_index']] = resultat
                                traitees += 1
                                erreurs += resultat['statut'] == 'erreur'
                                barre.update(1)
                                barre.set_postfix(erreurs=erreurs,
                                    chansons_s=f'{traitees / max(time.perf_counter() - debut, 1e-9):.2f}', refresh=False)
                except BaseException:
                    for tache in taches:
                        tache.cancel()
                    raise
        ordonnes = [resultats[position] for position in ids]
        if len(ordonnes) != len(selection):
            raise RuntimeError('Effectif final incohérent.')
        tableau = pd.DataFrame(ordonnes)
        tableau.to_csv(resultats_dir / f'{prefixe_sortie}.csv', index=False, encoding='utf-8-sig')
        (resultats_dir / f'{prefixe_sortie}.json').write_text(
            json.dumps(ordonnes, ensure_ascii=False, indent=2), encoding='utf-8')
        tableau.loc[tableau['statut'] == 'erreur'].to_csv(
            resultats_dir / f'{prefixe_sortie}_erreurs.csv', index=False, encoding='utf-8-sig')
        secondes = time.perf_counter() - debut
        metriques = {'perimetre': perimetre, 'modele': modele, 'empreinte': signature,
                     'selection': len(selection), 'reprises': reprises, 'traitees_session': traitees,
                     'reussites': int((tableau['statut'] == 'ok').sum()),
                     'erreurs': int((tableau['statut'] == 'erreur').sum()),
                     'secondes_session_client': secondes, 'chansons_s_session': traitees / secondes,
                     'secondes_par_chanson_session': secondes / traitees if traitees else None,
                     'date_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
                     'type_execution': 'reprise_cache' if not traitees else ('mixte' if reprises else 'calcul'),
                     'nombre_serveurs': len(base_urls),
                     'concurrence': concurrence, 'concurrence_par_serveur': concurrence_par_serveur,
                     'serveurs': json.dumps(base_urls), 'taille_lot': taille_lot, 'max_tokens': max_tokens,
                     'enable_thinking': enable_thinking,
                     'raisonnements_non_vides': int(tableau['raisonnement'].str.strip().ne('').sum())}
        pd.DataFrame([metriques]).to_csv(performance_dir / f'{prefixe}_performance.csv', index=False)
        performance_essai = pd.DataFrame([{
            'date': metriques['date_utc'], 'modele': modele, 'batch': taille_lot,
            'concurrent': concurrence, 'n': len(selection),
            'chansons/s': traitees / secondes if traitees else None,
        }])
        with cumul_path.open('a+', encoding='utf-8', newline='') as cumul:
            fcntl.flock(cumul, fcntl.LOCK_EX)
            cumul.seek(0, os.SEEK_END)
            vide = cumul.tell() == 0
            if not vide:
                cumul.seek(0)
                if list(pd.read_csv(cumul, nrows=0).columns) != list(performance_essai.columns):
                    raise ValueError(f'Colonnes incompatibles dans {cumul_path}. Corriger le fichier avant de relancer.')
            cumul.seek(0, os.SEEK_END)
            performance_essai.to_csv(cumul, index=False, header=vide)
            cumul.flush()
            os.fsync(cumul.fileno())
        valides = tableau.loc[tableau['statut'] == 'ok']
        comptes = valides['etiquette'].value_counts().reindex(source.ETIQUETTES, fill_value=0)
        distribution = pd.DataFrame([
            {'etiquette': e, 'effectif': comptes[e],
             'pourcentage_valides': 100 * comptes[e] / len(valides) if len(valides) else None}
            for e in source.ETIQUETTES])
        distribution.to_csv(performance_dir / f'{prefixe}_distribution.csv', index=False)
    display(pd.DataFrame([metriques]))
    display(distribution)
    if metriques['erreurs']:
        print('ATTENTION : erreurs techniques présentes ; voir le CSV erreurs et relancer pour les retenter.')
    print('Annotations :', resultats_dir / f'{prefixe_sortie}.csv')
    print('Performances cumulées :', cumul_path)
    return tableau


### 4.3 Lancer l'annotation de la sélection

Cette cellule annote **la sélection préparée à l'étape 3**, sans nouveau tirage : 220 chansons pour le test actuel, ou les 12 938 chansons du corpus d'étude avec `taille_echantillon = None`. Elle démarre directement, sans pilote ni approbation : vérifier le périmètre affiché à l'étape 3 avant de l'exécuter.

#### Contrôles techniques et lecture des résultats

Le contrôle `/v1/models` précède toute nouvelle requête d'annotation, sauf lors d'une reprise entièrement en cache. Chaque réussite exige un raisonnement séparé non vide et un JSON final strictement conforme. À la fin du traitement, les erreurs sont exportées et la cellule signale explicitement un lot incomplet si certaines chansons échouent. Les fichiers finaux contiennent toutes les lignes sélectionnées, erreurs comprises, dans l'ordre du corpus, avec leur année. Une relance traite uniquement les chansons encore absentes ou en erreur ; une reprise complète ne nécessite pas de serveur. Pour le calcul complet, une interruption n'est donc pas grave : relancer la cellule reprend là où le journal s'est arrêté.

Les exports se nomment `autoportrait_qwen_annotation_{n}.csv` et `.json` : `autoportrait_qwen_annotation_220` pour le test actuel, `autoportrait_qwen_annotation_12938` pour le corpus d'étude complet. L'aperçu est limité à 20 lignes pour ne pas saturer le carnet ; cette limite ne réduit pas les fichiers. Lire les décisions et les justifications, puis consulter `raisonnement` pour les cas ambigus. Vérifier que le modèle applique bien **AUTO + PORTRAIT**, et ne se fonde pas seulement sur la présence de « je ». Un raisonnement long et un taux de réussite technique de 100 % ne constituent pas une mesure de précision. Pour une proportion globale d'autoportraits, repondérer par année avec `chansons_exploitables` (étape 2).

In [7]:
resultats = executer_annotation(selection, perimetre)
display(resultats[['source_row_index', 'artiste', 'titre', 'annee', 'etiquette', 'justification', 'erreur']].head(20))
if not resultats['statut'].eq('ok').all():
    raise RuntimeError('Lot incomplet : consulter le fichier erreurs et relancer pour reprendre.')

strates : 1380 réussites reprises ; 11558 chansons à traiter.
Serveur disponible : http://127.0.0.1:8016/v1 | Qwen/Qwen3-32B
Serveur disponible : http://127.0.0.1:8017/v1 | Qwen/Qwen3-32B


Annotation strates:  11%|#         | 1380/12938 [00:00<?, ?chansons/s]

,perimetre,modele,empreinte,selection,reprises,traitees_session,reussites,erreurs,secondes_session_client,chansons_s_session,...,date_utc,type_execution,nombre_serveurs,concurrence,concurrence_par_serveur,serveurs,taille_lot,max_tokens,enable_thinking,raisonnements_non_vides
0,strates,Qwen/Qwen3-32B,1fb0edb5706b7a5ca988706576d2cd8019425dd507779a...,12938,1380,11558,12931,7,9551.404015,1.210084,...,2026-10-28T13:39:08Z,mixte,2,24,12,"[""http://127.0.0.1:8016/v1"", ""http://127.0.0.1...",128,8192,True,12931


,etiquette,effectif,pourcentage_valides
0,autoportrait,8861,68.525249
1,je_ne_sais_pas,183,1.415204
2,pas_autoportrait,3887,30.059547


ATTENTION : erreurs techniques présentes ; voir le CSV erreurs et relancer pour les retenter.
Annotations : /home/cbenavent/test/Rap/Autoportrait/data/results/autoportrait_qwen_annotation_12938.csv
Performances cumulées : /home/cbenavent/test/Rap/Autoportrait/data/performance/perfGPU_autoportrait.csv


,source_row_index,artiste,titre,annee,etiquette,justification,erreur
0,0,Dry,94310,2012,autoportrait,Le texte présente principalement celui qui par...,
1,4,Dry,14 ans déjà,2013,pas_autoportrait,Le texte parle surtout de la perte d'un ami (L...,
2,5,Lynda,A ma place,2014,autoportrait,Le texte se concentre sur l'artiste qui présen...,
3,18,PLK,1€ / 1Eu,2024,autoportrait,Le texte se concentre sur l'image de soi de ce...,
4,19,PLK,Une pêche une patate,2024,autoportrait,Le texte décrit principalement le rappeur lui-...,
5,28,Kohndo,Un Idéal,2006,pas_autoportrait,Le texte décrit surtout le monde et les diffic...,
6,29,Rim’K,Jour de pluie,2024,autoportrait,Le texte décrit principalement le rappeur lui-...,
7,32,Psy 4 de la Rime,Le goût de la honte,2008,pas_autoportrait,Le texte parle surtout du collectif des 'mal-a...,
8,33,Marjinal (FRA),No Love,2024,pas_autoportrait,Le texte parle surtout d'une relation amoureus...,
9,41,Wejdene,Promets moi,2024,pas_autoportrait,Le texte parle surtout d'une demande d'engagem...,


RuntimeError: Lot incomplet : consulter le fichier erreurs et relancer pour reprendre.